## Running LYNX

Conditional VGAE for integrated spatial trajectory reconstruction

In [ ]:
import os
import gc
import sys
import time

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import pyro
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from IPython.display import display

sns.set_context('paper')
rcParams.update({'font.family': 'Liberation Sans'})
rcParams.update({'font.size': 12})
rcParams.update({'figure.dpi': 180})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, plot, utils, trajectory
import vgae, configs, dataset
from importlib import reload

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%reload_ext autoreload

### Single-sample

In [ ]:
# Load paired Xenium & DESI
xenium_path = '../data/xenium/'
desi_path = '../data/desi/'
sample_id = 'NIH_F5'

adata_xenium = IO.load_xenium(os.path.join(xenium_path, sample_id))
adata_desi = sc.read_h5ad(os.path.join(desi_path, sample_id+'.h5'))
adata_xenium, adata_desi = IO.filter_cells(adata_xenium, adata_desi, by='map')

if 'cell_type' in adata_xenium.obs.keys():
    adata_xenium.obs['leiden'], categories = adata_xenium.obs.cell_type.factorize()
    categories = categories.values
else:
    adata_norm = adata_xenium.copy()
    sc.pp.normalize_total(adata_norm)
    sc.pp.log1p(adata_norm)

    sc.pp.pca(adata_norm)
    sc.pp.neighbors(adata_norm)
    sc.tl.leiden(adata_norm, random_state=42)
    adata_xenium.obs['leiden'] = adata_norm.obs['leiden'].copy()
    del adata_norm   


Load dataset:

In [ ]:
# Dataset specs
n_subgraphs = 16
k = 20
r = 50
sigma = 20

graph_data = dataset.HeteroDataset(
    adatas_ref=adata_xenium, 
    adatas_query=adata_desi,
    n_subgraphs=n_subgraphs, 
    k=k,
    r=r,
    is_weighted=True,
    sigma=sigma,
    verbose=True
)

train_data, val_data = random_split(graph_data, [0.7, 0.3])
train_dl, val_dl = DataLoader(train_data, shuffle=True), DataLoader(val_data)

Configure model:

In [ ]:
# Model parameters
n_hidden = 32
n_latent = 6

# Training parameters
n_epochs = 400
lr = 1e-2
patience = 20

# Training & Inference
train_configs = configs.set_train_configs(
    n_epochs=n_epochs, lr=lr, patience=patience, anneal=False,
    device=torch.device('cuda'),
)

model_configs = configs.set_model_configs(
    c_in=adata_xenium.shape[1],   # ref-dim 
    c_aux=adata_desi.shape[1],  # query-dim
    c_hidden=n_hidden, 
    c_latent=n_latent,
    act=nn.SiLU(),
    ref=graph_data.ref, 
    query=graph_data.query,
    num_clusters=graph_data.num_clusters,
    gene_symbols=adata_xenium.var_names,
) 


In [ ]:
pyro.clear_param_store()
torch.cuda.empty_cache()
gc.collect()

model = vgae.HeteroVGAE(model_configs, device=torch.device('cuda'))
model.fit(train_configs, train_dl=train_dl, val_dl=val_dl, DEBUG=True)

# Full inference with best model params
res = model.evaluate(
    adata_xenium, adata_desi,
    graph_data=graph_data,
    device=torch.device('cpu')
)


In [ ]:
# (1). Reconstruction
rand_indices = np.random.choice(
    np.arange(adata_xenium.shape[0]*adata_xenium.shape[1]), 10000, replace=False
)
plot.disp_kde_scatter(
    adata_xenium.X.A.flatten()[rand_indices],
    res.px.flatten()[rand_indices],
    xlabel=r"Ground-truth observation",
    ylabel=r"Reconstructed observation",
    title='Xenium feature reconstruction'
)
del rand_indices
gc.collect()

In [ ]:
# (2). Latent disentanglement
plot.disp_factor_corr(res.qzu)
# plot.disp_spatial_latents(adata_xenium, res.qzx, ncols=3)

In [ ]:
# TMP: Loadfrom saved file

# (1). load z
qz_x = np.load('../results/lynx_hetero_xenium_6_NIH_F5.npy')
qz_u = np.load('../results/lynx_hetero_desi_6_NIH_F5.npy')
adata_xenium.obsm['X_z'] = qz_x
adata_desi.obsm['X_z'] = qz_u

# (2). load both z & t (trajectory)
# lynx_df = pd.read_csv('../results/liver/ablation/lynx.csv', index_col=[0])
# adata_xenium.obs['t'] = lynx_df['t'].values
# adata_xenium.obsm['X_z'] = lynx_df.iloc[:, 1:].values

In [ ]:
# (3). Trajectory inference
# Low-dim gradients (u)
trajectory.compute_trajectory(
    adata_desi, 
    use_rep='X_z',
    n_neighbors=100,
    root_marker='Taurine '
)

sq.pl.spatial_scatter(
    adata_desi, color='t', 
    cmap='RdBu_r', size=1, img=False,
    title=r'Spatial Trajectory ($\gamma(t)$)'+'\nLYNX (DESI)'
)

# High-dim gradients (x)
trajectory.compute_trajectory(
    adata_xenium, 
    use_rep='X_z',
    n_neighbors=100,
    root_marker='DPT'
)

sq.pl.spatial_scatter(
    adata_xenium, color='t', 
    cmap='RdBu_r', size=20, img=False,
    title=r'Spatial Trajectory ($\gamma(t)$)'+'\nLYNX (Xenium)'
)


# plot.disp_trajectory(
#     adata_desi, cmap='RdBu_r',
#     title='Spatial Gradients\n LYNX (DESI)'
# )

# plot.disp_trajectory(
#     adata_xenium, cmap='RdBu_r',
#     title='Spatial Gradients\n LYNX (Xenium)'
# )


In [ ]:
utils.get_zonation_features(
    adata_xenium, adata_desi, n_zones=5, option='piecewise', sample_id=sample_id
)

sq.pl.spatial_scatter(
    adata_xenium, color='zone', img=False, size=20
)

In [ ]:
plot.disp_trajectory(
    adata_desi, cmap='RdBu_r',
    title='Spatial Gradients\n LYNX (DESI)'
)

plot.disp_trajectory(
    adata_xenium, cmap='RdBu_r',
    title='Spatial Gradients\n LYNX (Xenium)'
)

In [ ]:
# Save results
np.save('../results/liver/LYNX_xenium_{}.npy'.format(n_latent), res.qzx)
np.save('../results/liver/LYNX_desi_{}.npy'.format(n_latent), res.qzu)


#### Interaction modules

In [ ]:
import holoviews as hv
hv.extension('bokeh')

def plot_celltype_interaction(attn_df, amplitude=1):
    assert np.array_equal(attn_df.index, attn_df.columns)
    attn_score = attn_df.values
    cell_types = attn_df.columns

    graph = hv.Graph([
        (cell_types[i], cell_types[j], attn_score[i, j])
        for i in range(len(cell_types)-1) for j in range(i+1, len(cell_types))
    ], vdims=['weight'])
    labels = hv.Labels(graph.nodes, ['x', 'y'], 'index')

    graph = graph.opts(
        node_color='index', edge_color=hv.dim('weight')*amplitude, cmap='Category10',
        edge_cmap='Reds', edge_line_width=hv.dim('weight')*amplitude,
    )
    graph = (graph * labels.opts(text_font_size='10pt', text_color='black'))

    return graph

##### Attention-based cell-type interactions

In [ ]:
# Helper functions
def build_celltype_attention(
    adata_subset,
    categories,
    attn_key='v_attn',
    celltype_key='cell_type',
    agg='mean',
    normalize=False
):
    """
    Parameters
    ----------
    adata_subset : AnnData
        The subset of the full data.
    attn_key : str
        Key in adata_subset.obsm storing the attention matrix (n_cells x n_clusters).
    celltype_key : str
        The obs field containing the raw cell type strings for each cell.
    agg : str or callable
        Aggregation function for grouping by cell type. E.g., 'mean', 'sum', etc.
    normalize : boolean
        Whether to use Laplacian normalization

    Returns
    -------
    pd.DataFrame
        DataFrame of shape (#full_celltypes, #full_celltypes) with the cell types
        as both row and column labels.
    """

    row_labels = adata_subset.obs[celltype_key].values

    # Retrieve the attention matrix.
    attn_matrix = adata_subset.obsm[attn_key]

    # Build a DataFrame: rows are the "target" cell types (raw strings)
    # and columns correspond to the factorized cell type labels.
    df = pd.DataFrame(
        data=attn_matrix,
        index=row_labels,
        columns=categories,
    )

    # Aggregate attention values by the target cell type.
    df_agg = df.groupby(level=0).agg(agg)

    intersection = df_agg.index.intersection(categories)
    df_agg = df_agg.loc[intersection]
    df_agg = df_agg[df_agg.index]

    # Symmetrize the matrix by adding its transpose.
    df_agg = df_agg.add(df_agg.T, fill_value=0)

    # Normalize rows by their sums.
    row_sums = df_agg.sum(axis=1).replace(0, np.nan)
    M = df_agg.values
    d_inv_sqrt = 1.0 / np.sqrt(row_sums.values)  # shape: (#celltypes,)
    # Outer product scaling: M_norm[i,j] = M[i,j] / sqrt(rowSum[i]*rowSum[j])
    M_norm = d_inv_sqrt[:, None] * M * d_inv_sqrt[None, :]

    # Return the final DataFrame with full_categories as both index and columns.
    df = pd.DataFrame(M_norm if normalize else M, index=df_agg.index, columns=df_agg.columns)

    df.loc[(df!=0).any(axis=1)]
    
    return df


def get_binned_attention_by_celltype(adata, celltype_categories, n_bins=50):
   
    # Get the pseudotime values.
    t_vals = adata.obs['t']

    # Build a DataFrame from the attention matrix with columns as the cell type labels.
    attn_df = pd.DataFrame(
        data=adata.obsm['v_attn'],
        index=adata.obs.index,
        columns=celltype_categories
    )
    # Add pseudotime to the DataFrame.
    attn_df['t'] = t_vals

    # Convert to long format: each row corresponds to one cell and one attended cell type.
    df_long = attn_df.melt(
        id_vars='t',
        value_vars=celltype_categories,
        var_name='cell_type',
        value_name='attn'
    )

    # Sort by pseudotime.
    df_long.sort_values(by='t', inplace=True)

    # Create a 'bin' column by binning pseudotime into quantile bins.
    df_long['bin'] = pd.qcut(df_long['t'], q=n_bins, labels=False)

    # Group by 'bin' and 'cell_type' to compute the average attention in each bin.
    grouped = df_long.groupby(['bin', 'cell_type']).agg({
        'attn': 'mean',
        't': 'mean'  # optionally, keep the average pseudotime for plotting
    }).reset_index()

    return grouped

# Example usage with n_bins == 5 for illustration:
# bin_mean = get_binned_attention_by_celltype(adata_xenium, celltype_categories=categories, n_bins=50)

# # Plotting the average attention per cell type along pseudotime.
# unique_cell_types = bin_mean['cell_type'].unique()

# fig, axes = plt.subplots(
#     nrows=len(unique_cell_types),
#     ncols=1,
#     figsize=(6, 2 * len(unique_cell_types)),
#     sharex=True
# )

# for i, ctype in enumerate(unique_cell_types):
#     subdf = bin_mean[bin_mean['cell_type'] == ctype]
#     axes[i].plot(subdf['bin'], subdf['attn'], lw=2)
#     axes[i].set_ylabel('Average Attention')
#     axes[i].set_title(f'Cell Type: {ctype}')

# axes[-1].set_xlabel('Pseudotime Bins')
# plt.tight_layout()
# plt.show()



In [ ]:
attn_dfs = []
attn_graphs = []

for cluster_id in sorted(adata_xenium.obs['zone'].unique()):
    adata_sub = adata_xenium[adata_xenium.obs['zone'] == cluster_id].copy()
    zone_attn_df = build_celltype_attention(
        adata_subset=adata_sub,
        attn_key='v_attn',
        celltype_key='cell_type',
        categories=categories,
        agg='mean',
        normalize=True
    )
    attn_dfs.append(zone_attn_df)

    plt.figure(figsize=(6,5))
    sns.heatmap(zone_attn_df, cmap='magma')
    plt.title(f"Cell-type attention for z_cluster={cluster_id}")
    plt.xlabel("Cell type")
    plt.ylabel("Cell type")
    plt.tight_layout()
    plt.show()

    attn_graph = plot_celltype_interaction(zone_attn_df, amplitude=10)
    attn_graphs.append(attn_graph)



In [ ]:
holomap = hv.HoloMap({i: graph for i, graph in enumerate(attn_graphs)},  kdims='{}\nBin (PV->CV)'.format(sample_id))
holomap = holomap.opts(
    xaxis=None, yaxis=None, axiswise=True,
    width=500, height=500
) 

holomap

In [ ]:
hv.save(
    holomap,
    '../results/downstream/cell_types/{}_attn'.format(sample_id), fmt='widgets'
)

##### Cell-type co-localization

In [ ]:
colocalize_dfs = []
colocalize_graphs = []
cluster_key = 'cell_type'

for cluster_id in sorted(adata_xenium.obs['zone'].unique()):
    
    adata_sub = adata_xenium[adata_xenium.obs['zone'] == cluster_id].copy()
    sq.gr.spatial_neighbors(adata_sub, coord_type='generic', n_neighs=100, radius=50, )
    sq.gr.nhood_enrichment(adata_sub, cluster_key=cluster_key)

    enrich_score = adata_sub.uns[cluster_key+'_nhood_enrichment']['zscore']
    # enrich_score[enrich_score <= 1.65] = 0.

    # For comparison sake, scale to [0, 1] for now
    enrich_score = (enrich_score-enrich_score.min()) / (enrich_score.max()-enrich_score.min())

    enrich_score_df = pd.DataFrame(
        enrich_score,
        index=adata_sub.obs[cluster_key].cat.categories,
        columns=adata_sub.obs[cluster_key].cat.categories
    )
    colocalize_dfs.append(enrich_score_df)

    plt.figure(figsize=(6,5))
    sns.heatmap(enrich_score_df, cmap='magma')
    plt.title(f"Cell-type colocalization for z_cluster={cluster_id}")
    plt.xlabel("Cell type")
    plt.ylabel("Cell type")
    plt.tight_layout()
    plt.show()

    graph = plot_celltype_interaction(enrich_score_df, amplitude=1)
    colocalize_graphs.append(graph)

del cluster_id, enrich_score, enrich_score_df, graph, adata_sub

In [ ]:
holomap = hv.HoloMap({i: graph for i, graph in enumerate(colocalize_graphs)}, kdims='{}\nBin (PV->CV)'.format(sample_id))
holomap = holomap.opts(
    xaxis=None, yaxis=None, axiswise=True,
    width=500, height=500
) 

holomap

In [ ]:
hv.save(
    holomap,
    '../results/downstream/cell_types/{}_colocalization'.format(sample_id), fmt='widgets'
)

In [ ]:
# Comparison btw per-zone attention scores & colocalization scores
def get_CMD(A, B):
    """
    Measurement of correlation matrix distance
    Reference: https://ieeexplore.ieee.org/document/1543265
    """
    trace = np.trace(A@B)
    d = 1 - trace / (np.linalg.norm(A, ord='fro') * np.linalg.norm(B, ord='fro'))
    return d


n_cell_types = len(categories)

distances = [
    get_CMD(A.values, B.values)
    for A, B in zip(attn_dfs, colocalize_dfs)
]

baseline_distances = [
    get_CMD(np.random.rand(n_cell_types, n_cell_types), np.random.rand(n_cell_types, n_cell_types))
    for _ in range(len(attn_dfs))
]


plot_df = pd.DataFrame({
    'Correlation Matrix Distance': distances + baseline_distances,
    'Label': ['Attn vs. Colocalization']*len(attn_dfs) + ['Random']*len(attn_dfs)
})


sns.boxplot(data=plot_df, x='Label', y='Correlation Matrix Distance')
plt.show()

### All-samples

In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/desi/'
sample_ids = sorted([
    sample_id for sample_id in os.listdir(xenium_path)
    if os.path.isdir(os.path.join(xenium_path, sample_id))
])

Configure model:

In [ ]:
# Model parameters
n_hidden = 32
n_latent = 6

# Training parameters
n_epochs = 400
lr = 1e-2
patience = 20

# Graph parameters
n_subgraphs = 16
k = 20
r = 50
sigma = 20

# Training & Inference
train_configs = configs.set_train_configs(
    n_epochs=n_epochs, lr=lr, patience=patience, 
    device=torch.device('cuda'),
    anneal=False,
    verbose=True,
)

Full model training:

In [ ]:
# Iterate through individual samples
latents_ref = []
latents_query = []

for sample_id in sample_ids:
    print('Training on {}...'.format(sample_id))

    # ---------------
    # Loading data
    # ---------------
    
    adata_xenium = IO.load_xenium(os.path.join(xenium_path, sample_id), load_img=True)
    adata_desi = sc.read_h5ad(os.path.join(desi_path, sample_id+'.h5'))
    adata_xenium, adata_desi = IO.filter_cells(adata_xenium, adata_desi, by='map')

    if 'cell_type' in adata_xenium.obs.keys():
        adata_xenium.obs['leiden'] = adata_xenium.obs.cell_type.factorize()[0]
    else:
        adata_norm = adata_xenium.copy()
        sc.pp.normalize_total(adata_norm)
        sc.pp.log1p(adata_norm)
    
        sc.pp.pca(adata_norm)
        sc.pp.neighbors(adata_norm)
        sc.tl.leiden(adata_norm, random_state=42)
    
        adata_xenium.obs['leiden'] = adata_norm.obs['leiden'].copy()
        del adata_norm   

    graph_data = dataset.HeteroDataset(
        adatas_ref=adata_xenium, 
        adatas_query=adata_desi,
        n_subgraphs=n_subgraphs, 
        k=k,
        r=r,
        is_weighted=True,
        sigma=sigma,
        verbose=True
    )

    train_data, val_data = random_split(graph_data, [0.7, 0.3])
    train_dl, val_dl = DataLoader(train_data, shuffle=True), DataLoader(val_data)

    model_configs = configs.set_model_configs(
        c_in=adata_xenium.shape[1],   # ref-dim 
        c_aux=adata_desi.shape[1],  # query-dim
        c_hidden=n_hidden, 
        c_latent=n_latent,
        act=nn.SiLU(),
        ref=graph_data.ref, 
        query=graph_data.query,
        k_hop=1,
        num_heads=1,
        num_clusters=graph_data.num_clusters,
        verbose=True
    ) 

    # -----------------------
    # Training & Inference
    # -----------------------
    model = vgae.HeteroVGAE(model_configs, device=torch.device('cuda'))
    model.fit(train_configs, train_dl=train_dl, val_dl=val_dl)

    res = model.evaluate(
        adata_xenium, adata_desi,
        graph_data=graph_data,
        device=torch.device('cpu')
    )

    adata_xenium.obsm['X_z'] = res.qzx
    adata_desi.obsm['X_z'] = res.qzu

    latents_ref.append(res.qzx)
    latents_query.append(res.qzu)

    del model
    gc.collect()
    torch.cuda.empty_cache()
    pyro.clear_param_store()

    
    # ----------------
    # Visualization
    # ----------------
    
    # Factor disentanglement
    plot.disp_factor_corr(res.qzu)

    # Trajectory inference
    try:
        trajectory.compute_trajectory(
            adata_xenium, 
            use_rep='X_z',
            n_neighbors=100,
            root_marker='DPT'
        )

        trajectory.compute_trajectory(
            adata_desi, 
            use_rep='X_z',
            n_neighbors=100,
            root_marker='Taurine '
        )

        # plot.disp_trajectory(
        #     adata_desi, 
        #     cmap='RdBu',
        #     title='Spatial Gradients\n LYNX (DESI)'
        # )

        # disp_spatial_latents(adata_xenium)

        sq.pl.spatial_scatter(
            adata_xenium, color='t', 
            cmap='RdBu_r', size=20, img=False,
            title=r'Trajectory Pseudotime ($\gamma(t)$)'+'\nLYNX (Xenium)'
        )

        sq.pl.spatial_scatter(
            adata_desi, color='t', 
            cmap='RdBu_r', size=1, img=False,
            title=r'Trajectory Pseudotime ($\gamma(t)$)'+'\nLYNX (DESI)'
        )

        sc.pp.normalize_total(adata_xenium)
        sc.pp.log1p(adata_xenium)
        utils.get_zonation_features(    
            adata_xenium, adata_desi,
            n_zones=5, sample_id=sample_id,
            show=True
        )

        plt.show()
        time.sleep(5)
        
        # Saving results
        np.save('../results/lynx_hetero_xenium_{0}_{1}.npy'.format(n_latent, sample_id), adata_xenium.obsm['X_z'])
        np.save('../results/lynx_hetero_desi_{0}_{1}.npy'.format(n_latent, sample_id), adata_desi.obsm['X_z'])
                
    except ValueError:
        continue

    gc.collect()
    print('=============\n\n')

---